# Linear Regression from Scratch using Numpy

### TODO: Write this using Class with attributes and methods

## Dummy data 

In [1]:
import numpy as np

Without the column of ones on our hypothesis for theta zero, our model will be forced to pass through the origin (y-intercept = 0)

In [2]:
X_raw = np.arange(1, 4).reshape(3,1)
y =(np.arange(2,8,2) *10).reshape(-1,1) # ensure y is n x 1

X = np.hstack([np.ones((X_raw.shape[0],1)), X_raw])
m,n = X.shape
print(X,y)



# add the intercept term to the 


[[1. 1.]
 [1. 2.]
 [1. 3.]] [[20]
 [40]
 [60]]


## Testing out our model on a csv file.

In [3]:
import pandas as pd
data = pd.read_csv("Salary_Data.csv")
# print(data)

X_raw = data.drop(columns=["Salary"]).values
X = np.hstack([np.ones((X_raw.shape[0],1)),X_raw])
y= data[["Salary"]].values

m,n = X.shape
print(X,y)

[[ 1.   1.1]
 [ 1.   1.3]
 [ 1.   1.5]
 [ 1.   2. ]
 [ 1.   2.2]
 [ 1.   2.9]
 [ 1.   3. ]
 [ 1.   3.2]
 [ 1.   3.2]
 [ 1.   3.7]
 [ 1.   3.9]
 [ 1.   4. ]
 [ 1.   4. ]
 [ 1.   4.1]
 [ 1.   4.5]
 [ 1.   4.9]
 [ 1.   5.1]
 [ 1.   5.3]
 [ 1.   5.9]
 [ 1.   6. ]
 [ 1.   6.8]
 [ 1.   7.1]
 [ 1.   7.9]
 [ 1.   8.2]
 [ 1.   8.7]
 [ 1.   9. ]
 [ 1.   9.5]
 [ 1.   9.6]
 [ 1.  10.3]
 [ 1.  10.5]] [[ 39343.]
 [ 46205.]
 [ 37731.]
 [ 43525.]
 [ 39891.]
 [ 56642.]
 [ 60150.]
 [ 54445.]
 [ 64445.]
 [ 57189.]
 [ 63218.]
 [ 55794.]
 [ 56957.]
 [ 57081.]
 [ 61111.]
 [ 67938.]
 [ 66029.]
 [ 83088.]
 [ 81363.]
 [ 93940.]
 [ 91738.]
 [ 98273.]
 [101302.]
 [113812.]
 [109431.]
 [105582.]
 [116969.]
 [112635.]
 [122391.]
 [121872.]]


In Gradient Descent, if your features and targets are not scaled to a similar, small range (usually between -1 and 1, or mean 0 and variance 1), Gradient Descent will crash and overflow. 

when i tested with large dataset with alpha = 0.01 and 0.1, my predictions where `NaNs`, as the GD made very huge steps. my initial cost was 9.6trillion

$$\theta_{\text{new}} = \theta - \alpha \times (\text{huge gradient})$$

It's gonna subtract a massive number from theta, making the weight to jump to infinity.

SO the solution is to reduce the learning rate or you scale the values.


In [4]:
# 1. Standardize X (Features: Years of Experience)
# Formula: (x - mean) / standard_deviation


# X_raw = (X_raw - np.mean(X_raw)) / np.std(X_raw)
#
# y = (y - np.mean(y)) / np.std(y)
#
# X = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])

In [5]:
def train_test_split(X,y,test_ratio=0.2,random_state=42):
    rng = np.random.default_rng(seed=random_state)

    num_sample = X.shape[0]
    test_size = int(num_sample * test_ratio)
    train_size = num_sample-test_size

    indices = np.arange(num_sample)

    rng.shuffle(indices)

    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    X_train,X_test = X[train_indices], X[test_indices]
    y_train,y_test = y[train_indices], y[test_indices]

    return X_train,X_test, y_train,y_test

X_train,X_test,y_train,y_test=train_test_split(X,y,test_ratio=0.2)

## Implemeting the Normal Equation

In [6]:
def normal_equation(X,y):
    return np.linalg.inv(X.T@X) @ X.T @ y

In [7]:
one_shot_weight = normal_equation(X_train,y_train)
print(one_shot_weight)

[[25005.96669946]
 [ 9614.24581246]]


In [8]:
def compute_cost(X,y,theta):
    """
    X: feature matrix: shape (m x n)
    y: target values: shape (m x 1 or m,)
    theta: model weights: shape (n x 1)
    """

    #calcualting the hypothesis h(x) = X . theta
    predictions = X @ theta
    #calculating hte error:
    errors = predictions - y
    return 0.5 * np.sum(errors ** 2)

# Gradient Descent

In [9]:
# batch gradient descent
def gradient_descent_step(theta, X, y,alpha=0.01 ):
    """
    alpha: learning rate
    theta: old model weight, shape (m x 1 or m,)
    X: feature matrix: shape (m x n)
    y: target values: shape (m x 1 or m,)
    """
    #calcualting the hypothesis h(x) = X . theta
    predictions = X @ theta
    #calculating hte error:
    errors = predictions - y #shape: m x 1
    theta_new = theta - alpha * X.T @ errors # instead of using loops, but the dimension for X (mxn) is not compaittible with errors (mx1), so we transpose X and dot it with errors to get same result and this is doen accross all rows 
    return theta_new

# Learning and Weight Update

In [10]:
EPOCHS = 200000
LEARNING_RATE = 1e-3
np.random.seed(42)

best_theta = None
best_epoch = 0
best_cost = float("inf")

theta = np.random.randn(n,1)
cost_history = []

for epoch in range(EPOCHS):
    theta = gradient_descent_step(theta, X_train, y_train,LEARNING_RATE)
    cost = compute_cost(X_train, y_train, theta)
    cost_history.append(cost)

    #check for improvemnt
    if cost < best_cost:
        best_cost = cost
        best_epoch = epoch
        best_theta = theta

print("Trained Weights (Gradient Descent):\n", theta)
print("Initial Cost:", cost_history[0])
print("Final Cost:", cost_history[-1])


print(f"Best Epoch: {best_epoch}")
print(f"Best Cost: {best_cost}")
print(f"Optimal Weights:\n {best_theta}")

prediction = X_test @ theta
print(f"==============\nModel Prediction =====\nActual:\n{y_test}\n\t\nPrediction:\n{prediction}") 


Trained Weights (Gradient Descent):
 [[25005.96669946]
 [ 9614.24581246]]
Initial Cost: 1502405446.7901096
Final Cost: 337936334.5151051
Best Epoch: 6128
Best Cost: 337936334.51510465
Optimal Weights:
 [[25005.96669913]
 [ 9614.24581251]]
Model Prediction =====
Actual:
[[ 37731.]
 [ 39891.]
 [112635.]
 [ 46205.]
 [ 57081.]
 [ 64445.]]
	
Prediction:
[[ 39427.33541815]
 [ 46157.30748687]
 [117302.72649906]
 [ 37504.48625566]
 [ 64424.37453054]
 [ 55771.55329933]]


# Rewriting all these to class approach

While research this, i learnt that LinearRegression class from sklearn does not make use of the iterative method of gradient descent but the normal equation.

So SGDRegressor (Stochastic Gradient Descent Regressor), makes use of the iterative approach

In [26]:
class LinearRegression:
    def __init__(self):
        self.theta = None
        self.best_theta = None

    def _compute_cost(self, X,y, theta):
        """
        X: feature matrix: shape (m x n)
        y: target values: shape (m x 1 or m,)
        theta: model weights: shape (n x 1)
        """

        #calcualting the hypothesis h(x) = X . theta
        predictions = X @ theta
        #calculating hte error:
        errors = predictions - y
        return 0.5 * np.sum(errors ** 2)
    
    def _normal_equation(self, X, y):
        return np.linalg.inv(X.T@X) @ X.T @ y


    def fit(self, X_train_raw,y_train):
        # automatcially handles the intercept column of ones so that the user won't have to

        m = X_train_raw.shape[0] # number of rows
        print(X_train_raw.shape)

        X_train = np.hstack((np.ones((m,1)),X_train_raw))

        n = X_train.shape[1] # number of features

        one_shot_weight = self._normal_equation(X_train,y_train)
        self.theta = one_shot_weight

        return self

    def predict(self,X_test_raw):
        if not self.theta.any():
            raise ValueError("This model has not been fitted yet! Call .fit() first.")

        m = X_test_raw.shape[0]
        X_test = np.hstack((np.ones((m, 1)), X_test_raw))

        return X_test @ self.theta

In [28]:
data = pd.read_csv("Salary_Data.csv")
# print(data)

X = data.drop(columns=["Salary"]).values
y= data[["Salary"]].values



X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42)
print(X_test.shape,y.shape)

l_model = LinearRegression()

l_model.fit(X_train,y_train)

y_pred = l_model.predict(X_test)
print("Predictions:\n", y_pred)



(6, 1) (30, 1)
(24, 1)
Predictions:
 [[ 39427.33541815]
 [ 46157.30748687]
 [117302.72649906]
 [ 37504.48625566]
 [ 64424.37453054]
 [ 55771.55329933]]


In [75]:

class BGDRegressor:
    def __init__(self,learning_rate=1e-4,epochs=20000,random_state=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.theta = None
        self.best_theta = None
        self.best_cost = float("inf")
        self.best_epoch = 0
        self.cost_history = [ ]
        self.seed=random_state

    def _compute_cost(self, X,y, theta):
        """
        X: feature matrix: shape (m x n)
        y: target values: shape (m x 1 or m,)
        theta: model weights: shape (n x 1)
        """

        #calcualting the hypothesis h(x) = X . theta
        predictions = X @ theta
        #calculating hte error:
        errors = predictions - y
        return 0.5 * np.sum(errors ** 2)
    
    def _normal_equation(self, X, y):
        return np.linalg.inv(X.T@X) @ X.T @ y

    def _batch_gradient_descent_step(self, X,y):
        """
        X: feature matrix: shape (m x n)
        y: target values: shape (m x 1 or m,)
        """

        predictions = X @ self.theta
        errors = predictions - y

        # Gradient calculation
        gradient = X.T @ errors
        
        return self.theta - (self.learning_rate * gradient) 


    def fit(self, X_train_raw,y_train):
        # automatcially handles the intercept column of ones so that the user won't have to

        rng = np.random.default_rng(self.seed)
        m = X_train_raw.shape[0] # number of rows
        X_train = np.hstack((np.ones((m,1)),X_train_raw))

        self.theta = rng.random(size=(X_train.shape[1],1))

        n = X_train.shape[1] # number of features


        for epoch in range(self.epochs):
            self.theta = self._batch_gradient_descent_step(X_train,y_train )

            cost = self._compute_cost(X_train, y_train, self.theta)
            self.cost_history.append(cost)

        # Track best parameters
            if cost < self.best_cost:
                self.best_cost = cost
                self.best_epoch = epoch
                self.best_theta = self.theta.copy()

        print(f"Training Complete! Best Cost: {self.best_cost:.4f} at Epoch {self.best_epoch}")
        return self



    def predict(self,X_test_raw):
        if not self.theta.any():
            raise ValueError("This model has not been fitted yet! Call .fit() first.")

        m = X_test_raw.shape[0]
        X_test = np.hstack((np.ones((m, 1)), X_test_raw))

        return X_test @ self.best_theta
    

In [76]:
bgd_model = BGDRegressor(learning_rate=1e-3,epochs=200000)

bgd_model.fit(X_train,y_train)

y_pred=bgd_model.predict(X_test)
print(y_pred)
print(y_test)

Training Complete! Best Cost: 337936334.5151 at Epoch 5750
[[ 39427.33541694]
 [ 46157.30748582]
 [117302.72649966]
 [ 37504.4862544 ]
 [ 64424.37452991]
 [ 55771.5532985 ]]
[[ 37731.]
 [ 39891.]
 [112635.]
 [ 46205.]
 [ 57081.]
 [ 64445.]]
